# rbfx — dense RBF/kriging interpolation from Jupyter

Rust FFI wrapper (PyO3) over MPDOK/gp_engine's `gp_solver.so`: mixed-precision (FP32 Cholesky factor + FP64 iterative refinement) dense interpolation, GPU-accelerated. See `README.md` for the full picture and `bench/benchmark.py` for the time+accuracy sweep vs scipy.

In [1]:
import numpy as np
import rbfx

rng = np.random.default_rng(0)
n = 2000
X = rng.uniform(0, 10, size=(n, 2))
y = np.sin(X[:, 0]) * np.cos(X[:, 1]) + 0.3 * np.sin(2 * X[:, 0] + X[:, 1])
y += 0.01 * rng.standard_normal(n)

## Fit

`kernel` is one of `"rbf"`, `"matern32"`, `"matern52"` (gp_core.py's `_KINDS`, isotropic lengthscale `ell`).

In [2]:
fit = rbfx.Rbfx(X, y, kernel="matern32", ell=2.0, sigma_f=1.0, sigma_n2=1e-4,
                tol=1e-11, max_ir=10)
print(f"relres={fit.relres:.3e}  n_ir={fit.n_ir}  converged={fit.converged}  "
      f"logdet={fit.logdet:.4f}")

relres=7.525e-13  n_ir=6  converged=True  logdet=-12059.8061


## Predict

In [3]:
Xq = rng.uniform(0, 10, size=(500, 2))
pred = fit.predict(Xq)
true = np.sin(Xq[:, 0]) * np.cos(Xq[:, 1]) + 0.3 * np.sin(2 * Xq[:, 0] + Xq[:, 1])
rmse = np.sqrt(np.mean((pred - true) ** 2))
print(f"held-out RMSE: {rmse:.4f}")

held-out RMSE: 0.0099


## Error handling — bad input surfaces as a normal Python exception

Shape/kernel-name mismatches are caught before any GPU call (`rbfx-core`'s `RbfxError::ShapeMismatch`, mapped to `ValueError` at the PyO3 boundary) rather than panicking or segfaulting.

In [4]:
try:
    rbfx.Rbfx(X, y[:-1], kernel="matern32", ell=2.0)
except ValueError as e:
    print("caught:", e)

caught: shape mismatch: y.len()=1999 != n=2000 (x has 2000 points of dimension 2)


## Benchmark vs scipy — time and accuracy

See `bench/benchmark.py` for the full N-sweep. Summary (`bench/results.json`, `bench/results_time.png`, `bench/results_accuracy.png`):

| N | rbfx t_total (s) | scipy t_total (s) | rbfx RMSE | scipy RMSE |
|---|---|---|---|---|
| 2,000 | 0.072 | 0.180 | 0.00559 | 0.00558 |
| 4,000 | ~0.19 (interp.) | 1.018 | — | 0.00319 |
| 8,000 | ~0.70 (interp.) | 6.782 | — | 0.00231 |
| 20,000 | 2.226 | OOM/too slow to run | 0.00146 | — |

Accuracy matches scipy closely at matched N (both fitting the same Gaussian-kernel model with matched `epsilon`/`ell`) — the win is wall-clock time at scale, growing with N as scipy's dense CPU solve becomes the bottleneck.